# 최종 제출 재현 코드 — 난임 임신성공 예측 (해커톤 제출용)

이 노트북을 **위에서 아래로 한 번 실행**하면 최종 리더보드 제출 파일 `submission.csv`가 재생성됩니다.

**최종 모델 — 5개 멤버 랭크 블렌드**
`lgb(v2v3) · cat(v2v3) · xgb(v3) · lin(ratio) · **tabicl(v2)**`  ← NN 자리를 TabICLv2로 교체한 비교용 변형, 가중치는 OOF(train 라벨만) 힐클라이밍.

> 📌 **비교용 변형**: 원본(NN) 노트북과 이 노트북(TabICLv2)을 각각 실행해 `블렌드 OOF AUC`(와 필요시 LB)를 비교하면, 4개 멤버가 동일하므로 NN↔TabICLv2 교체 효과만 분리해서 볼 수 있습니다.

**규정 준수 (test 누수 차단)**
- test 데이터는 **어떤 학습에도 들어가지 않습니다.** 명목형 인코딩·스케일·결측 대치·타깃인코딩은 전부 **fold 내부(train fold)에서만 fit** 후 valid/test에 transform.
- 외부 데이터·유사라벨링(pseudo-labeling) 미사용. 단일 결정적 실행.

**입력:** `train.csv` · `test.csv` · `sample_submission.csv`  
**환경:** ★ GPU 세션 필요(TabICLv2). 트리·선형은 CPU.  
**산출:** `submission.csv` (+ `reproduce_report.json`, `env_versions.json`, 멤버 OOF/test = 누수안전 증빙)

In [17]:
# ============================================================
# 재현성 고정 (시드 · 결정성 · 버전 스냅샷)
# ============================================================
import os, random
import numpy as np
SEED = 42
os.environ["PYTHONHASHSEED"]="42"; os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"; os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
random.seed(SEED); np.random.seed(SEED)
NUM_THREADS = 4   # 트리 결정성 전제. 재실행 시 동일 유지.
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True)   # 엄격 결정성(임베딩 backward까지 결정적 경로 강제). 동일 환경서 NN bit-재현.
except Exception: pass
def snapshot_env(path="env_versions.json"):
    import json,sys,sklearn,scipy,pandas as pd
    v={"python":sys.version.split()[0],"numpy":np.__version__,"pandas":pd.__version__,
       "scikit-learn":sklearn.__version__,"scipy":scipy.__version__}
    for n in ["lightgbm","xgboost","catboost","torch"]:
        try: v[n]=__import__(n).__version__
        except Exception: v[n]=None
    try: json.dump(v,open(path,"w"),indent=2)
    except Exception: pass
    print("[env]",v); return v
ENV=snapshot_env()

[env] {'python': '3.12.13', 'numpy': '2.4.6', 'pandas': '2.3.3', 'scikit-learn': '1.6.1', 'scipy': '1.16.3', 'lightgbm': '4.6.0', 'xgboost': '3.2.0', 'catboost': '1.2.10', 'torch': '2.10.0+cu128'}


## 1. 데이터 로드 + 누수안전 피처 빌더 (행단위·train-only)

In [18]:
import glob, re, json
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier

def find_csv(n):
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]
    assert h, f"{n} 없음 — Add Input 확인"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train)
print("train",train.shape,"| test",test.shape,"| base_rate=%.4f"%y.mean())

def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def runr(x): return rankdata(x)/len(x)
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]

# ── 트리 베이스 (명목형 카테고리 코드는 train 카테고리로만 fit) ──
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}); return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    ts=df[COL_PROC].apply(_tp)
    for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]  # pandas3.0: str dtype 대비(is_numeric_dtype)
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
st=fit_tree(train); Xb,CATF=tf_tree(train,st); Xb_te,_=tf_tree(test,st); Xb_te=Xb_te.reindex(columns=Xb.columns)
base_num=Xb.drop(columns=CATF); base_num_te=Xb_te.drop(columns=CATF)

# ── v2 게이팅 파생 (행단위) ──
def masks(df):
    return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,
            "ICSI":NUM(df,"미세주입된 난자 수")>0,
            "본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def build_v2_gated(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"])
    F["g본인_난자수율"]=P6.where(Mk["본인난자"]); F["g기증_난자수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"])
    F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    F["FZ3_해동난자수율"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"해동 난자 수"))
    F["PG1_PGT강도"]=NUM(df,"착상 전 유전 검사 사용 여부").fillna(0)+NUM(df,"착상 전 유전 진단 사용 여부").fillna(0)
    F["PG2_PGT분류"]=NUM(df,"PGD 시술 여부").fillna(0)+NUM(df,"PGS 시술 여부").fillna(0)
    F["ST1_자극"]=NUM(df,"배란 자극 여부").fillna(0)
    return pd.DataFrame(F,index=df.index)
V2tr=build_v2_gated(train); V2te=build_v2_gated(test)

# ── 선형용 비율 파생 (행단위) ──
def build_lin_ratios(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["P1_수정률"]=P1; F["P2_ICSI수정"]=P2; F["P6_난자수율"]=P6; F["L3_배양일수"]=L3
    F["P3_활용률"]=DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["P5_저장률"]=DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["L6_배아perICSI난자"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수")); F["N6_난자감모"]=DIV(NUM(df,"수집된 신선 난자 수")-NUM(df,"혼합된 난자 수"),NUM(df,"수집된 신선 난자 수"))
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"]); F["g본인_수율"]=P6.where(Mk["본인난자"]); F["g기증_수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"]); F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
RATtr=build_lin_ratios(train); RATte=build_lin_ratios(test)

# ── v3 신규 파생 (행단위) ──
def build_new_derived(df):
    F={}
    tx=NUM(df,"이식된 배아 수").fillna(0); sto=NUM(df,"저장된 배아 수").fillna(0); emb=NUM(df,"총 생성 배아 수").fillna(0)
    ses=(df["단일 배아 이식 여부"]==1).values
    F["EL_set_type"]=np.where(~ses,0,np.where(sto.values>0,2,1)).astype("int8")
    F["FA_no_transfer"]=(tx==0).astype("int8").values
    is_current=df[COL_RSN].astype(str).str.contains("현재 시술용",na=False).values
    F["FA_nontransfer_reason"]=(~is_current).astype("int8")
    return pd.DataFrame(F,index=df.index)
Dtr=build_new_derived(train); Dte=build_new_derived(test)

# ── v3 트리 후보 컬럼 (최종 선택 셋) ──
#   C1=배아이식 유형(EL_set_type), C2=무이식 플래그(FA_*).  C3(ICSI 경로효율)은 혼합난자⊇미세주입난자로 분모 오정의 → 영구 폐기.
V3_TREE_COLS = ["EL_set_type","FA_no_transfer","FA_nontransfer_reason"]
assert all(c in Dtr.columns for c in V3_TREE_COLS)
# 행단위 독립성(누수 0) 검증
assert np.array_equal(build_new_derived(train.head(300)).fillna(-9).values, Dtr.head(300).fillna(-9).values), "행단위 독립성 위반!"
print("피처 빌더 준비 ✅ | tf_tree",Xb.shape,"| v2게이팅",V2tr.shape[1],"| 비율",RATtr.shape[1],"| v3트리",len(V3_TREE_COLS))

train (256351, 69) | test (90067, 68) | base_rate=0.2583
피처 빌더 준비 ✅ | tf_tree (256351, 72) | v2게이팅 11 | 비율 15 | v3트리 3


## 2. 트리·선형 멤버 학습기 (fold-내부 fit · 결정적)

In [19]:
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,
         feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50,
         deterministic=True,force_row_wise=True,num_threads=NUM_THREADS)
XGP=dict(objective="binary:logistic",eval_metric="auc",tree_method="hist",learning_rate=0.05,
         max_depth=6,subsample=0.8,colsample_bytree=0.8,nthread=NUM_THREADS)
TREE_ITERS=1500
def fit_one(kind,Xt,yt,Xv,yv,catf,seed):
    if kind=="lgb": return lgb.train(dict(LGP,seed=seed),lgb.Dataset(Xt,yt,categorical_feature=catf),TREE_ITERS,
        valid_sets=[lgb.Dataset(Xv,yv,categorical_feature=catf)],callbacks=[lgb.early_stopping(80,verbose=False),lgb.log_evaluation(0)])
    if kind=="xgb": return xgb.train(dict(XGP,seed=seed),xgb.DMatrix(Xt.values,label=yt),TREE_ITERS,
        evals=[(xgb.DMatrix(Xv.values,label=yv),"v")],early_stopping_rounds=80,verbose_eval=False)
    m=CatBoostClassifier(iterations=TREE_ITERS,learning_rate=0.05,depth=6,verbose=0,random_seed=seed,
        early_stopping_rounds=80,thread_count=NUM_THREADS); m.fit(Xt,yt,eval_set=(Xv,yv),cat_features=catf); return m
def pred_one(kind,m,X):
    if kind=="lgb": return m.predict(X)
    if kind=="xgb": return m.predict(xgb.DMatrix(X.values),iteration_range=(0,m.best_iteration+1))
    return m.predict_proba(X)[:,1]
def tree_member(kind, extra_tr, extra_te, seed=42):
    """단일 트리 모델 OOF + test (extra_* = 추가 파생 행렬). fold-내부 fit."""
    X  =pd.concat([Xb.reset_index(drop=True),    extra_tr.reset_index(drop=True)],axis=1)
    Xte=pd.concat([Xb_te.reset_index(drop=True), extra_te.reset_index(drop=True)],axis=1)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(X,y)); catf=[c for c in CATF if c in X.columns]
    o=np.zeros(N); tt=np.zeros(len(Xte))
    for tri,vai in folds:
        m=fit_one(kind,X.iloc[tri],y[tri],X.iloc[vai],y[vai],catf,seed); o[vai]=pred_one(kind,m,X.iloc[vai])
        tt+=pred_one(kind,m,Xte)/len(folds)
    return o,tt

# 선형 (타깃인코딩·스케일 전부 fold-내부 fit)
TE_COLS=NOMINAL_COLS+[COL_PROC,COL_RSN]
def te_fit(cat,yy,sm=20):
    g=pd.DataFrame({"c":cat.values,"y":yy}).groupby("c")["y"].agg(["mean","count"]); pr=float(yy.mean())
    return ((g["mean"]*g["count"]+pr*sm)/(g["count"]+sm)),pr
def lin_member(use_ratios=True,seed=42):
    oof=np.zeros(N); tst=np.zeros(len(test))
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(base_num,y):
        Xt=base_num.iloc[tri].copy(); Xv=base_num.iloc[vai].copy(); Xte=base_num_te.copy()
        for c in TE_COLS:
            enc,pr=te_fit(train[c].astype(str).iloc[tri],y[tri])
            Xt[f"te_{c}"]=train[c].astype(str).iloc[tri].map(enc).fillna(pr).values
            Xv[f"te_{c}"]=train[c].astype(str).iloc[vai].map(enc).fillna(pr).values
            Xte[f"te_{c}"]=test[c].astype(str).map(enc).fillna(pr).values
        if use_ratios:
            Xt=pd.concat([Xt.reset_index(drop=True),RATtr.iloc[tri].reset_index(drop=True)],axis=1)
            Xv=pd.concat([Xv.reset_index(drop=True),RATtr.iloc[vai].reset_index(drop=True)],axis=1)
            Xte=pd.concat([Xte.reset_index(drop=True),RATte.reset_index(drop=True)],axis=1)
        med=Xt.median(); Xt=Xt.fillna(med); Xv=Xv.fillna(med)
        sc=StandardScaler().fit(Xt); m=LogisticRegression(max_iter=2000,C=0.5).fit(sc.transform(Xt),y[tri])
        oof[vai]=m.predict_proba(sc.transform(Xv))[:,1]
        tst+=m.predict_proba(sc.transform(Xte.fillna(med)))[:,1]/5
    return oof,tst
print("트리·선형 학습기 준비 ✅")

트리·선형 학습기 준비 ✅


## 3. TabDPT 멤버 — fold-0 홀드아웃 OOF만 (다른 멤버는 5-fold 그대로)

TabDPT는 retrieval 기반 제로샷 — 쿼리마다 최근접 이웃만 context로 써서 가벼움. fold 0에서만 OOF.
test 예측은 11번에서 게이트(증분 CI>0) 통과 시에만 전체-train context로 1회.

In [20]:
# ============================================================
# TabDPT — fold-0 홀드아웃 OOF (retrieval 제로샷, 메모리 가벼움). 다른 멤버는 5-fold 그대로.
#   · 누수: test는 어떤 context에도 미포함. 전처리는 context(train) 쪽에서만 fit.
# ============================================================
import subprocess, sys, gc, time
subprocess.run([sys.executable,"-m","pip","install","-q","tabdpt"], check=False)
from tabdpt import TabDPTClassifier
import torch
try: torch.use_deterministic_algorithms(False)
except Exception: pass

def _free():
    for _v in ("tclf","Xt_td","Xv_td","Xall_td","Xte_td"):
        if _v in globals():
            try: del globals()[_v]
            except Exception: pass
    gc.collect()
    try:
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception: pass

def _td_clean(df):
    df=df.drop(columns=[c for c in [TARGET,ID_COL] if c in df.columns]).copy()
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    return df
_TD_TR=_td_clean(train); _TDCOLS=list(_TD_TR.columns)
_TDOBJ=[c for c in _TDCOLS if not pd.api.types.is_numeric_dtype(_TD_TR[c])]
_TDNUM=[c for c in _TDCOLS if c not in _TDOBJ]

def _td_prep(fit_df, dfs):
    F=_td_clean(fit_df).reindex(columns=_TDCOLS)
    keep=[c for c in _TDCOLS if F[c].nunique(dropna=True)>1]            # 상수컬럼 제거
    objk=[c for c in _TDOBJ if c in keep]; numk=[c for c in _TDNUM if c in keep]
    enc={c:{v:i for i,v in enumerate(sorted(F[c].astype(str).fillna("NA").unique()))} for c in objk}
    med={}; clip={}
    for c in numk:
        s=pd.to_numeric(F[c],errors="coerce"); mm=s.median(); med[c]=float(mm) if np.isfinite(mm) else 0.0
        lo,hi=s.quantile(0.001),s.quantile(0.999)
        clip[c]=(float(lo) if np.isfinite(lo) else med[c], float(hi) if np.isfinite(hi) else med[c])
    out=[]
    for df in dfs:
        D=_td_clean(df).reindex(columns=_TDCOLS); cc=[]
        for c in objk: cc.append(D[c].astype(str).fillna("NA").map(enc[c]).fillna(-1).astype(np.float32).values)
        for c in numk:
            x=pd.to_numeric(D[c],errors="coerce").fillna(med[c]); lo,hi=clip[c]
            if hi>lo: x=x.clip(lo,hi)
            cc.append(x.astype(np.float32).values)
        out.append(np.column_stack(cc).astype(np.float32))
    return out

def _td_clf():
    # retrieval: 쿼리마다 최근접 context_size개만 context로 → 메모리/속도 가벼움.
    # T4(Turing) 호환 위해 flash attention·torch.compile off (필요시 True로).
    return TabDPTClassifier(context_reduction="retrieval", device=None,
                            use_flash=False, compile=False, verbose=False)

os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"  # 단편화 완화
def _predict_chunked(clf, X, tag="", chunk=5000):
    out = np.empty(len(X)); t0 = time.perf_counter(); nC = (len(X)+chunk-1)//chunk
    for ci in range(nC):
        s = ci*chunk; e = min(s+chunk, len(X))
        out[s:e] = clf.predict_proba(X[s:e], context_size=CTX, batch_size=BATCH, seed=SEED)[:,1]
        el = time.perf_counter()-t0; done = e/len(X); eta = el/done*(1-done)
        print(f"    {tag} {done*100:4.0f}% ({e}/{len(X)}) | 경과 {el/60:.1f}분 | ETA ~{eta/60:.0f}분", flush=True)
    return out

CTX   = 512    # 더 빠르게: 검색·attention(512²) 가벼움. 프로브엔 충분
BATCH = 128    # CTX 512라 메모리 여유 → 배치 키워 속도↑
_folds=list(StratifiedKFold(5,shuffle=True,random_state=SEED).split(train,y))
tri,vai=_folds[0]

_free(); _t0=time.perf_counter()
Xt_td,Xv_td=_td_prep(train.iloc[tri],[train.iloc[tri],train.iloc[vai]])   # context = fold0-train만
print(f"  TabDPT 유효피처 {Xt_td.shape[1]} (상수 제거 후)")
tclf=_td_clf(); tclf.fit(Xt_td,y[tri])                                    # ★ context = fold0-train ONLY
oof_tabdpt=np.full(N,np.nan)
oof_tabdpt[vai]=_predict_chunked(tclf, Xv_td, tag='holdout')   # fold0 valid에만 채움 (진행률 출력)
print(f"tabdpt 홀드아웃 OOF AUC = {roc_auc_score(y[vai],oof_tabdpt[vai]):.5f}  |  {(time.perf_counter()-_t0)/60:.1f}분")
del tclf,Xt_td,Xv_td; _free()


  TabDPT 유효피처 62 (상수 제거 후)
    holdout   10% (5000/51271) | 경과 3.1분 | ETA ~29분
    holdout   20% (10000/51271) | 경과 6.2분 | ETA ~26분
    holdout   29% (15000/51271) | 경과 9.3분 | ETA ~23분
    holdout   39% (20000/51271) | 경과 12.4분 | ETA ~19분
    holdout   49% (25000/51271) | 경과 15.6분 | ETA ~16분
    holdout   59% (30000/51271) | 경과 18.7분 | ETA ~13분
    holdout   68% (35000/51271) | 경과 21.8분 | ETA ~10분
    holdout   78% (40000/51271) | 경과 24.9분 | ETA ~7분
    holdout   88% (45000/51271) | 경과 28.0분 | ETA ~4분
    holdout   98% (50000/51271) | 경과 31.1분 | ETA ~1분
    holdout  100% (51271/51271) | 경과 31.9분 | ETA ~0분
tabdpt 홀드아웃 OOF AUC = 0.72181  |  31.9분


## 4. 5개 멤버 생성 (seed42 · 동일 fold)

In [21]:
members_oof={}; members_test={}
# lgb(v2v3) · cat(v2v3) = tf_tree base + v2 게이팅 + v3   (그대로 5-fold)
ev2v3_tr=pd.concat([V2tr.reset_index(drop=True),Dtr[V3_TREE_COLS].reset_index(drop=True)],axis=1)
ev2v3_te=pd.concat([V2te.reset_index(drop=True),Dte[V3_TREE_COLS].reset_index(drop=True)],axis=1)
for kind in ["lgb","cat"]:
    o,t=tree_member(kind,ev2v3_tr,ev2v3_te,seed=42); members_oof[kind]=o; members_test[kind]=t
    print(f"  {kind}(v2v3) OOF={roc_auc_score(y,o):.5f}")
# xgb(v3) = tf_tree base + v3 (v2 미포함)
o,t=tree_member("xgb",Dtr[V3_TREE_COLS],Dte[V3_TREE_COLS],seed=42); members_oof["xgb"]=o; members_test["xgb"]=t
print(f"  xgb(v3)  OOF={roc_auc_score(y,o):.5f}")
# lin(ratio) = base + TE + 비율
o,t=lin_member(use_ratios=True,seed=42); members_oof["lin"]=o; members_test["lin"]=t
print(f"  lin(ratio) OOF={roc_auc_score(y,o):.5f}")
# tabdpt — fold-0 홀드아웃 OOF만 (test는 11번에서 게이트 통과 시 전체-context로 생성)
members_oof["tabdpt"]=oof_tabdpt
print(f"  tabdpt 홀드아웃 OOF={roc_auc_score(y[vai],oof_tabdpt[vai]):.5f}")

for m in members_oof:
    pd.DataFrame({f"oof_{m}":members_oof[m],"y":y}).to_csv(f"oof_{m}.csv",index=False)
for m in members_test:
    pd.DataFrame({"ID":test[ID_COL].values,f"test_{m}":members_test[m]}).to_csv(f"test_{m}.csv",index=False)
print("멤버 생성 완료 ✅ (트리·선형 5-fold + tabdpt 홀드아웃)")


  lgb(v2v3) OOF=0.73965


KeyboardInterrupt: 

## 5. 최종 블렌드 → `submission.csv`

In [ ]:
from scipy.stats import spearmanr
def hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get)
    ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm},best[1]

BASE=["lgb","cat","xgb","lin"]
for m in BASE: assert roc_auc_score(y,members_oof[m])<0.999, f"{m} 누수 의심"
vmask=np.isfinite(members_oof["tabdpt"]); ys=y[vmask]      # = fold0 valid
assert roc_auc_score(ys,members_oof["tabdpt"][vmask])<0.999, "tabdpt 누수 의심"

# 4멤버 full-OOF 가중치 (안전 기본)
R4={m:runr(members_oof[m]) for m in BASE}
w4,oof4=hill(R4,y)
print(f"[4멤버 full-OOF] 블렌드 OOF = {oof4:.5f}")

# fold-0 슬라이스에서 tabdpt 증분 측정
Rs ={m:runr(members_oof[m][vmask]) for m in BASE}
Rsp=dict(Rs, tabdpt=runr(members_oof["tabdpt"][vmask]))
wb,ab=hill(Rs,ys); wp,ap=hill(Rsp,ys)
print(f"[fold0 슬라이스] 4멤버 {ab:.5f} → +tabdpt {ap:.5f}  (Δ={ap-ab:+.5f})")
print("  ρ:", {k:round(spearmanr(members_oof['tabdpt'][vmask],members_oof[k][vmask]).correlation,3) for k in BASE})
rng=np.random.RandomState(SEED); n=len(ys)
pb=sum(wb[k]*Rs[k] for k in wb if wb[k]>0); pp=sum(wp[k]*Rsp[k] for k in wp if wp[k]>0)
d=np.empty(2000)
for i in range(2000):
    bi=rng.randint(0,n,n); d[i]=roc_auc_score(ys[bi],pp[bi])-roc_auc_score(ys[bi],pb[bi])
lo,hi=np.percentile(d,[2.5,97.5])
INCLUDE_TABDPT = lo>0
print(f"  증분 95% CI=[{lo:+.5f},{hi:+.5f}] → {'tabdpt 포함 ✅' if INCLUDE_TABDPT else 'tabdpt 제외(4멤버)'}")

if INCLUDE_TABDPT:
    print("→ 전체-context test 예측 (전체 train을 context로, retrieval) ...")
    _free(); _t0=time.perf_counter()
    Xall_td,Xte_td=_td_prep(train,[train,test])          # 전처리 fit on FULL train, transform train+test
    tclf=_td_clf(); tclf.fit(Xall_td,y)                   # context = 전체 train (test 미포함=누수0)
    members_test["tabdpt"]=_predict_chunked(tclf, Xte_td, tag='test')
    del tclf,Xall_td,Xte_td; _free()
    print(f"  완료 {(time.perf_counter()-_t0)/60:.1f}분")
    w=wp; p_test=sum(w[m]*runr(members_test[m]) for m in w if w[m]>0)
else:
    w=w4; p_test=sum(w[m]*runr(members_test[m]) for m in w if w[m]>0)

assert not np.isnan(p_test).any()
sp=[p for p in glob.glob("/kaggle/input/**/sample_submission.csv",recursive=True)]
if sp:
    s=pd.read_csv(sp[0]); pc=[c for c in s.columns if c.lower()!="id"][0]; s[pc]=p_test
else:
    s=pd.DataFrame({"ID":test[ID_COL].values,"probability":p_test}); pc="probability"
s.to_csv("submission.csv",index=False)
print(f"💾 submission.csv | n={len(s)} | tabdpt {'포함' if INCLUDE_TABDPT else '제외'}")
print("   가중치 →", {k:round(v,3) for k,v in w.items()})
